In [ ]:
import os
import glob
import pandas as pd
import numpy as np


def compute_all_models_table(file_paths_or_dir, output_csv="model_comparison_summary.csv"):
    # 1. Resolve file list
    if isinstance(file_paths_or_dir, str):
        if os.path.isdir(file_paths_or_dir):
            file_list = glob.glob(os.path.join(file_paths_or_dir, "*.csv"))
        else:
            file_list = glob.glob(file_paths_or_dir)
    else:
        file_list = file_paths_or_dir

    if not file_list:
        raise FileNotFoundError("No CSV files found matching the provided path or list.")

    # 2. Sort files to match Table 2 order: ResNet -> U-Net -> SegNet
    order_priority = {"resnet": 1, "unet": 2, "segnet": 3}

    def get_sort_key(path):
        name = os.path.basename(path).lower()
        for key, val in order_priority.items():
            if key in name:
                return val
        return 99

    sorted_files = sorted(file_list, key=get_sort_key)

    # 3. Flexible column mapping for validation metrics
    metric_patterns = {
        "Accuracy": ["val_accuracy", "val_accu", "val_acc"],
        "Dice": ["val_dice"],
        "IoU": ["val_iou", "val_jaccard"],
        "Precision": ["val_precision", "val_prec"],
        "Recall": ["val_recall", "val_reca", "val_rec", "val_sensitivity"],
        "F1-Score": ["val_f1", "val_f1_score", "val_f1-score"],
    }

    all_rows = []

    for filepath in sorted_files:
        if not os.path.exists(filepath):
            continue

        df = pd.read_csv(filepath)
        filename = os.path.splitext(os.path.basename(filepath))[0]

        # Determine Model name display
        if "resnet" in filename.lower():
            model_name = "ResNet"
        elif "unet" in filename.lower():
            model_name = "U-Net"
        elif "segnet" in filename.lower():
            model_name = "SegNet"
        else:
            model_name = filename.replace("_1000", "").replace("_1800", "").replace("_", "-").capitalize()

        # Filter out non-numeric summary rows (e.g., 'AVERAGE' or 'mean')
        fold_cols = [c for c in df.columns if c.lower() in ["fold", "kfold", "folds", "split"]]
        if fold_cols:
            fold_col = fold_cols[0]
            df = df[pd.to_numeric(df[fold_col], errors="coerce").notnull()].copy()

        lower_to_orig = {col.lower(): col for col in df.columns}
        metrics_summary = {}

        for metric, candidates in metric_patterns.items():
            matched_col = None
            for cand in candidates:
                if cand in lower_to_orig:
                    matched_col = lower_to_orig[cand]
                    break

            # In binary segmentation, F1-Score is identical to Dice
            if matched_col is None and metric == "F1-Score":
                for cand in metric_patterns["Dice"]:
                    if cand in lower_to_orig:
                        matched_col = lower_to_orig[cand]
                        break

            if matched_col is not None:
                series = pd.to_numeric(df[matched_col], errors="coerce").dropna()
                mean = series.mean()
                std = series.std(ddof=1)  # Sample standard deviation across folds
                metrics_summary[metric] = f"{mean:.4f} ± {std:.4f}"
            else:
                metrics_summary[metric] = "N/A"

        # Assemble row dictionary
        row = {
            "Model": model_name,
            "Accuracy": metrics_summary["Accuracy"],
            "Dice": metrics_summary["Dice"],
            "IoU": metrics_summary["IoU"],
            "Precision": metrics_summary["Precision"],
            "Recall": metrics_summary["Recall"],
            "F1-Score": metrics_summary["F1-Score"],
        }
        all_rows.append(row)

    table_df = pd.DataFrame(all_rows)

    # Optional: Save summary to CSV
    if output_csv:
        table_df.to_csv(output_csv, index=False)

    return table_df


# ========================================================
# Execution
# ========================================================
if __name__ == "__main__":
    files_to_process = ["resnet_1000.csv", "unet_1000.csv", "segnet_1000.csv"]

    summary_df = compute_all_models_table(
        files_to_process, 
        output_csv="model_comparison_summary.csv"
    )

    print("=" * 95)
    print("Table 2: Aggregated Model Performance (Mean ± Std)")
    print("=" * 95)
    print(summary_df.to_string(index=False))
    print("=" * 95)
    print("\nSummary saved to: model_comparison_summary.csv")